# EXPLORATION - NOT MAIN PORTFOLIO
# DATE: 10/3/26

# FINDINGS TO BE ADDED TO MAIN:

## Goal
Testing several parameters for the model selected on the previous experiment to find the best parameter for the model. Final outcome is to decide which parameter to use for the final model

## Setup

In [1]:
import pandas as pd

data = pd.read_csv('creditcard.csv')

data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


as decided in previous experiments, we will remove some of the v features, use SMOTE resampling, and we use random forest for the model.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, \
confusion_matrix

from imblearn.over_sampling import SMOTE

from datetime import datetime

In [3]:
#remove v
low_score_v = ['V22', 'V23', 'V25', 'V26', 'V28', 'V15', 'V24', 'V13', 'V27']

reduced_v = data.drop(columns=low_score_v)
X = reduced_v.drop('Class', axis=1)
y = reduced_v['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

#SMOTE resample
smote_sample = SMOTE(random_state=1)
X_train, y_train = smote_sample.fit_resample(X_train, y_train)

In [4]:
X_train.shape

(454902, 21)

## Tuning

### Baseline

the baseline for comparison will use the simplest parameter as used in exp 2 & 3. Model with tuned parameter will be compared to this baseline

In [5]:
base_model = RandomForestClassifier(n_estimators=100, random_state=1, n_jobs=-1)

start_time = time.time()
base_model.fit(X_train, y_train)
baseline_time = time.time() - start_time

y_pred_base = base_model.predict(X_test)

print('Baseline Performance')
print(classification_report(y_test, y_pred_base))
print('Confusion Matrix')
print(confusion_matrix(y_test,y_pred_base))

baseline_metrics = {
    'precision': precision_score(y_test, y_pred_base),
    'recall': recall_score(y_test, y_pred_base),
    'f1': f1_score(y_test, y_pred_base),
    'train_time': baseline_time
}

Baseline Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.88      0.89      0.88        98

    accuracy                           1.00     56962
   macro avg       0.94      0.94      0.94     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56852    12]
 [   11    87]]


In [6]:
print(f"\nBaseline Performance:")
print(f"  Precision:     {baseline_metrics['precision']:.4f}")
print(f"  Recall:        {baseline_metrics['recall']:.4f}")
print(f"  F1-Score:      {baseline_metrics['f1']:.4f}")
print(f"  Training Time: {baseline_metrics['train_time']:.2f}s")


Baseline Performance:
  Precision:     0.8788
  Recall:        0.8878
  F1-Score:      0.8832
  Training Time: 66.23s


baseline performance is already good enough but we will see if hyperparameter tuning can improve the performance even more. Even an improvement as slightly as 1% is already good enough to be consider

### Hyperparameter Tuning
#### Parameters to Tune:
1. **n_estimators**: 100, 200, 300, 500
2. **max_depth**: 15, 20, 25, 30, None
3. **min_samples_split**: 2, 5, 10, 15
4. **max_features**: 'sqrt', 'log2', None
5. **bootstrap**: True, False

We will use GridSearchCV to do the tuning

In [7]:
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [15, 20, 25, 30, None],
    'min_samples_split': [2, 5, 10, 15],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [ ]:
rf_grid = GridSearchCV(
    estimator = RandomForestClassifier(random_state=1, n_jobs=-1),
    param_grid = param_grid,
    cv=3,
    scoring='recall',
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

# Train (this takes a while!)
print(f"\nStarting grid search at {datetime.now().strftime('%H:%M:%S')}...")

start_time = time.time()
rf_grid.fit(X_train, y_train)
grid_search_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"Grid search complete! Time taken: {grid_search_time/60:.1f} minutes")


Starting grid search at 08:09:55...
Fitting 3 folds for each of 480 candidates, totalling 1440 fits


In [ ]:
print('Best Parameter:')
for param, value in rf_grid.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest Cross-Validation Recall Score: {rf_grid.best_score_:.4f}")
print(f"Baseline Recall Score:               {baseline_metrics['recall']:.4f}")
print(f"Improvement:                     {rf_grid.best_score_ - baseline_metrics['recall']:+.4f}")

### Evaluate on Test Set

In [ ]:
best_rf = rf_grid.best_estimator_

y_pred_tuned = best_rf.predict(X_test_scaled)

print('Tuned Performance')
print(classification_report(y_test, y_pred_tuned))
print('Tuned Confusion Matrix')
print(confusion_matrix(y_test,y_pred_tuned))

tuned_metrics = {
    'precision': precision_score(y_test, y_pred_tuned),
    'recall': recall_score(y_test, y_pred_tuned),
    'f1': f1_score(y_test, y_pred_tuned)
}

In [ ]:
print("Tuned Model Performance:")
print(f"  Precision:  {tuned_metrics['precision']:.4f}")
print(f"  Recall:     {tuned_metrics['recall']:.4f}")
print(f"  F1-Score:   {tuned_metrics['f1']:.4f}")

In [ ]:
rf_grid.best_params_

I've tried grid search several times and I realized my current machine is not really the best for hyperparameter tuning as it takes too long to even finish one. I've tried from 9000 fits to around 1000 fits and still no luck. From here on, I'll just do random search to find the best params and probably do grid search with value near the best parameter for validation.

### Hyperparameter Tuning #2

#### Parameters to Tune:
1. **n_estimators**: 100, 200, 300, 500
2. **max_depth**: 15, 20, 25, 30, None
3. **min_samples_split**: 2, 5, 10, 15
4. **max_features**: 'sqrt', 'log2', None
5. **bootstrap**: True, False

We will use RandomizedSearchCV to do the tuning

In [7]:
param_random = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [15, 20, 25, 30, None],
    'min_samples_split': [2, 5, 10, 15],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [ ]:
rf_random = RandomizedSearchCV(
    estimator = RandomForestClassifier(random_state=1, n_jobs=-1),
    param_distributions = param_random,
    n_iter=40,
    cv=3,
    scoring='recall',
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    random_state=1
)

# Train (this takes a while!)
print(f"\nStarting random search at {datetime.now().strftime('%H:%M:%S')}...")

start_time = time.time()
rf_random.fit(X_train, y_train)
random_search_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"Random search complete! Time taken: {random_search_time/60:.1f} minutes")


Starting random search at 15:16:18...
Fitting 3 folds for each of 40 candidates, totalling 120 fits


In [ ]:
print('Best Parameter:')
for param, value in rf_random.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest Cross-Validation Recall Score: {rf_random.best_score_:.4f}")
print(f"Baseline Recall Score:               {baseline_metrics['recall']:.4f}")
print(f"Improvement:                     {rf_random.best_score_ - baseline_metrics['recall']:+.4f}")

#### Validate with GridSearch

In [ ]:
print(rf_random.best_params_)

In [5]:
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [15, 20, 25, 30, None],
    'min_samples_split': [2, 5, 10, 15],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [ ]:
rf_grid = GridSearchCV(
    estimator = RandomForestClassifier(random_state=1, n_jobs=-1),
    param_grid = param_grid,
    cv=3,
    scoring='recall',
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)